In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import pandas as pd
import nltk
from nltk import word_tokenize
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

In [2]:
device = torch.device("cuda")

In [3]:
from google.colab import files
uploaded = files.upload()

Saving IMDB_dataset.csv to IMDB_dataset.csv


In [4]:
df = pd.read_csv('IMDB_dataset.csv')

In [5]:
df.drop_duplicates(inplace = True)

In [6]:
df['review'] = df['review'].str.lower()

# removing url
def remove_url(text):
    return re.sub(r'http\S+', '', text)

df['review'] = df['review'].apply(remove_url)

In [7]:
# removing punctuation
def remove_punct(text):
    return re.sub(r'[^A-Za-z0-9\s]','', text)

df['review'] = df['review'].apply(remove_punct)

# remove html tags
def remove_tag(text):
    return re.sub(r'<.*?>', '', text)

df['review'] = df['review'].apply(remove_tag)

In [8]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [9]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words('english')

    for token in tokens:
        if token in stop_words:
            text = text.replace(token,'')

    return text

df['review'] = df['review'].apply(remove_stopwords)


def stemming(text):
    tokens = word_tokenize(text)
    stemmed_tokens = []
    ps = PorterStemmer()

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_tokens.append(stemmed_token)

    return ' '.join(stemmed_tokens)


df['review'] = df['review'].apply(stemming)


In [10]:
le =  LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])

In [52]:
import pickle

tk = TfidfVectorizer(max_features = 5000)
X = tk.fit_transform(df['review'])

pickle.dump(tk, open("vectorizer.pkl", "wb"))

In [12]:
X_train, X_test, Y_train, Y_test = train_test_split(X, df['sentiment'], test_size = 0.2, random_state = 42)

X_train = X_train.toarray()
X_test = X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(Y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(Y_test.values).float()
)


train_loader = DataLoader(train_set, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_set, batch_size = 64)

In [13]:
class RNN(nn.Module):

    def __init__(self, input_size, hidden_size = 128, num_of_layers = 1):

        super().__init__()

        self.hidden_size = hidden_size
        self.num_of_layers = num_of_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_of_layers, batch_first = True)

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_of_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.rnn(x, h0) # (batch_size, seq_size, hidden_size)
        out = self.fc(out[:,-1,:])

        return out

In [14]:
input_size = X_train.shape[1]

model = RNN(input_size)
model = model.to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [15]:
#unsqueeze (give additional dimensiton) and squeeze(remove dimension)
epochs = 10

for epoch in range(epochs):
    model.train()

    running_loss = 0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        xb = xb.unsqueeze(1)
        output = model.forward(xb)
        output = torch.sigmoid(output.squeeze())

        loss = criterion(output, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    running_test_loss = 0

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
      for xb, yb in test_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        xb = xb.unsqueeze(1)
        output = model.forward(xb)
        output = torch.sigmoid(output.squeeze())
        loss = criterion(output, yb)
        running_test_loss += loss.item()

        pred = (output > 0.5).float()
        correct += (pred == yb).sum().item()
        total += output.size(0)

    print(f'epoch {epoch}/{epochs} loss train = {running_loss/len(train_loader)} and loss test = {running_test_loss/len(test_loader)} accuracy: {correct/total}')

epoch 0/10 loss train = 0.3790670986617765 and loss test = 0.2999358015675699 accuracy: 0.869617827972169
epoch 1/10 loss train = 0.26399499265657317 and loss test = 0.30629644230488806 accuracy: 0.8680044368256529
epoch 2/10 loss train = 0.24906747225792178 and loss test = 0.31989268766295526 accuracy: 0.8625592417061612
epoch 3/10 loss train = 0.24128846913095445 and loss test = 0.32118323781797964 accuracy: 0.8603408288797015
epoch 4/10 loss train = 0.2372032800388913 and loss test = 0.32524904647181113 accuracy: 0.8602399919330442
epoch 5/10 loss train = 0.2339538761084118 and loss test = 0.32782104111486865 accuracy: 0.8592316224664717
epoch 6/10 loss train = 0.23160567347320818 and loss test = 0.3329378079983496 accuracy: 0.8583240899465564
epoch 7/10 loss train = 0.2305954730198268 and loss test = 0.33149012375262477 accuracy: 0.8576182313199556
epoch 8/10 loss train = 0.22861079356122402 and loss test = 0.33360072191684476 accuracy: 0.8561056771200968
epoch 9/10 loss train = 0.

In [16]:
torch.save(model.state_dict(),'sentiment_state_dict.pt')

In [17]:
device = torch.device('cpu')
model.load_state_dict(torch.load('sentiment_state_dict.pt',map_location=device, weights_only=False))


<All keys matched successfully>

In [43]:
model = model.to(device)
model.eval()

correct = 0
total = 0
with torch.no_grad():
    for xb, yb in test_loader:

        xb = xb.to(device)
        yb = yb.to(device)
        xb = xb.unsqueeze(1)
        print(xb.shape)
        output = model.forward(xb)
        output = torch.sigmoid(output.squeeze())

        pred = (output > 0.5).float()

        correct += (pred == yb).sum().item()
        total += yb.size(0)

    print('accuracy:',correct/total)

torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([64, 1, 5000])
torch.Size([

In [47]:
def test_preprocessing(text):

    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)
    text = re.sub(r'<.*?>', '', text)

    tokens = word_tokenize(text)
    stop_words = stopwords.words('english')

    tokens = [t for t in tokens if t not in stop_words]

    ps = PorterStemmer()
    tokens = [ps.stem(t) for t in tokens]

    text = ' '.join(tokens)

    X = tk.transform([text])

    return X


In [50]:
text = 'This is a bad movie, you must not watch it.<br/>'
text_matrix = test_preprocessing(text)
print(text_matrix.shape)

(1, 5000)


In [51]:
model = model.to(device)
model.eval()
xb = torch.tensor(text_matrix.toarray(), dtype = torch.float32)

with torch.no_grad():
      xb = xb.to(device)
      xb = xb.unsqueeze(1)
      print(xb.shape)
      output = model(xb)
      output = torch.sigmoid(output.squeeze())

      pred = (output > 0.5).float()

print(pred)

torch.Size([1, 1, 5000])
tensor(0.)


In [40]:
print(xb)

tensor([[[0.4082],
         [0.4082],
         [0.4082],
         [0.4082],
         [0.4082],
         [0.4082]]])
